In [ ]:
import pandas as pd
import os
from google.colab import drive
# <<pip install BioPython>>
from Bio import SeqIO

drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/Senior Year/Undergraduate Research Program/cafa-6-protein-function-prediction"
os.listdir(DATA_DIR)

train_terms = pd.read_csv(f"{DATA_DIR}/Train/train_terms.tsv", sep="\t")

sequences = {r.id.split("|")[1]: str(r.seq) for r in SeqIO.parse(f"{DATA_DIR}/Train/train_sequences.fasta", "fasta")}

ia = pd.read_csv(f"{DATA_DIR}/IA.tsv", sep="\t", header=None, names=["term", "IA"])


Mounted at /content/drive


In [ ]:
term_counts = train_terms['term'].value_counts()
frequent_terms = term_counts[term_counts >= 50].index
filtered_terms = train_terms[train_terms['term'].isin(frequent_terms)]

In [ ]:
seq_df = pd.DataFrame({'EntryID': list(sequences.keys()), 'sequence': list(sequences.values())})
seq_df.head()

,EntryID,sequence
0,A0A0C5B5G6,MRWQEMGYIFYPRKLR
1,A0JNW5,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...
2,A0JP26,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...
3,A0PK11,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...
4,A1A4S6,MGLQPLEFSDCYLDSPWFRERIRAHEAELERTNKFIKELIKDGKNL...


In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU: Tesla T4


In [ ]:
from transformers import AutoModel, AutoTokenizer
import numpy as np
from tqdm import tqdm

# Load ESM-2 model
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to("cuda").eval()

print("Model loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


In [ ]:
# Prepare sequences as a list
protein_ids = list(sequences.keys())
protein_seqs = list(sequences.values())

print(f"Generating embeddings for {len(protein_ids)} proteins...")

batch_size = 8
max_length = 1024
all_embeddings = []

with torch.no_grad():
    for i in tqdm(range(0, len(protein_seqs), batch_size)):
        batch_seqs = protein_seqs[i:i+batch_size]

        # Truncate long sequences
        batch_seqs = [s[:max_length] for s in batch_seqs]

        # Tokenize
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True,
                          truncation=True, max_length=max_length+2).to("cuda")

        # Get embeddings
        outputs = model(**inputs)

        # Mean pool across sequence length (ignore padding)
        attention_mask = inputs["attention_mask"].unsqueeze(-1)
        hidden = outputs.last_hidden_state
        masked = hidden * attention_mask
        embeddings = masked.sum(dim=1) / attention_mask.sum(dim=1)

        all_embeddings.append(embeddings.cpu().numpy())

# Combine all batches
embeddings_matrix = np.concatenate(all_embeddings, axis=0)
print(f"Done! Shape: {embeddings_matrix.shape}")

Generating embeddings for 82404 proteins...


  0%|          | 11/10301 [00:42<11:09:46,  3.91s/it]


KeyboardInterrupt: 

In [ ]:
np.save(f"{DATA_DIR}/embeddings_matrix.npy", embeddings_matrix)
np.save(f"{DATA_DIR}/protein_ids.npy", np.array(protein_ids))